In [1]:
import math

# ---------------- DATASET ----------------

data = [
    ["Sunny", "Hot", "High", "No"],
    ["Sunny", "Hot", "Normal", "Yes"],
    ["Overcast", "Hot", "High", "Yes"],
    ["Rain", "Mild", "High", "Yes"],
    ["Rain", "Cool", "Normal", "Yes"],
    ["Rain", "Cool", "High", "No"],
    ["Overcast", "Cool", "Normal", "Yes"],
    ["Sunny", "Mild", "High", "No"],
    ["Sunny", "Cool", "Normal", "Yes"],
    ["Rain", "Mild", "Normal", "Yes"]
]

attributes = ["Outlook", "Temperature", "Humidity"]


# ---------------- PRINT DATASET ----------------

print("DATASET")
print("-" * 60)

print("Outlook\tTemperature\tHumidity\tPlay")

for row in data:
    print("\t".join(row))


# ---------------- ENTROPY ----------------

def entropy(data):

    total = len(data)

    yes = sum(1 for row in data if row[-1] == "Yes")
    no = total - yes

    if yes == 0 or no == 0:
        return 0

    p_yes = yes / total
    p_no = no / total

    return -(p_yes * math.log2(p_yes) +
             p_no * math.log2(p_no))


# ---------------- INFORMATION GAIN ----------------

def information_gain(data, attribute_index):

    total_entropy = entropy(data)

    values = set(row[attribute_index] for row in data)

    weighted_entropy = 0

    for value in values:

        subset = [
            row for row in data
            if row[attribute_index] == value
        ]

        weighted_entropy += (
            len(subset) / len(data)
        ) * entropy(subset)

    return total_entropy - weighted_entropy


# ---------------- PRINT INFORMATION GAIN ----------------

print("\nENTROPY")
print("-" * 40)

print("Dataset Entropy =", round(entropy(data), 4))


print("\nINFORMATION GAIN")
print("-" * 40)

gains = []

for i in range(len(attributes)):

    gain = information_gain(data, i)

    gains.append(gain)

    print(
        attributes[i],
        "=",
        round(gain, 4)
    )


# ---------------- ID3 ----------------

def id3(data, attribute_indices):

    classes = [row[-1] for row in data]

    # If all belong to same class
    if classes.count(classes[0]) == len(classes):
        return classes[0]

    # If no attributes remain
    if len(attribute_indices) == 0:
        return max(set(classes), key=classes.count)

    # Find best attribute
    gains = [
        information_gain(data, i)
        for i in attribute_indices
    ]

    best_index = attribute_indices[
        gains.index(max(gains))
    ]

    tree = {
        attributes[best_index]: {}
    }

    values = set(
        row[best_index]
        for row in data
    )

    remaining_attributes = [
        i for i in attribute_indices
        if i != best_index
    ]

    for value in values:

        subset = [
            row for row in data
            if row[best_index] == value
        ]

        tree[attributes[best_index]][value] = id3(
            subset,
            remaining_attributes
        )

    return tree


# ---------------- BUILD TREE ----------------

tree = id3(
    data,
    list(range(len(attributes)))
)


print("\nFINAL DECISION TREE")
print("-" * 40)

print(tree)


# ---------------- PREDICTION ----------------

def predict(tree, sample):

    # If leaf node
    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))

    attribute_index = attributes.index(attribute)

    value = sample[attribute_index]

    subtree = tree[attribute].get(value)

    if subtree is None:
        return "Unknown"

    return predict(subtree, sample)


# ---------------- TEST DATA ----------------

test_data = [
    ["Sunny", "Cool", "High"],
    ["Sunny", "Cool", "Normal"],
    ["Rain", "Cool", "High"],
    ["Overcast", "Hot", "High"]
]


print("\nPREDICTIONS")
print("-" * 50)

for sample in test_data:

    result = predict(tree, sample)

    print(
        sample,
        "=>",
        result
    )|

SyntaxError: invalid syntax (1355903895.py, line 211)

In [5]:


import pandas as pd
import math

# --------------------------------------------------
# STEP 1: Create Dataset
# --------------------------------------------------

data = {
    'Outlook': [
        'Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain',
        'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain',
        'Sunny', 'Overcast', 'Overcast', 'Rain'
    ],

    'Temperature': [
        'Hot', 'Hot', 'Hot', 'Mild', 'Cool',
        'Cool', 'Cool', 'Mild', 'Cool', 'Mild',
        'Mild', 'Mild', 'Hot', 'Mild'
    ],

    'Humidity': [
        'High', 'High', 'High', 'High', 'Normal',
        'Normal', 'Normal', 'High', 'Normal', 'Normal',
        'Normal', 'High', 'Normal', 'High'
    ],

    'Wind': [
        'Weak', 'Strong', 'Weak', 'Weak', 'Weak',
        'Strong', 'Strong', 'Weak', 'Weak', 'Weak',
        'Strong', 'Strong', 'Weak', 'Strong'
    ],

    'Play': [
        'No', 'No', 'Yes', 'Yes', 'Yes',
        'No', 'Yes', 'No', 'Yes', 'Yes',
        'Yes', 'Yes', 'Yes', 'No'
    ]
}

df = pd.DataFrame(data)

print("DATASET")
print(df)


# --------------------------------------------------
# STEP 2: Calculate Entropy
# --------------------------------------------------

def entropy(data, target):

    values = data[target].value_counts()

    total = len(data)

    ent = 0

    for count in values:

        probability = count / total

        ent -= probability * math.log2(probability)

    return ent


# --------------------------------------------------
# STEP 3: Calculate Information Gain
# --------------------------------------------------

def information_gain(data, attribute, target):

    total_entropy = entropy(data, target)

    weighted_entropy = 0

    for value in data[attribute].unique():

        subset = data[data[attribute] == value]

        weight = len(subset) / len(data)

        weighted_entropy += weight * entropy(
            subset,
            target
        )

    gain = total_entropy - weighted_entropy

    return gain


# --------------------------------------------------
# STEP 4: Find Best Attribute
# --------------------------------------------------

def best_attribute(data, attributes, target):

    gains = {}

    for attribute in attributes:

        gains[attribute] = information_gain(
            data,
            attribute,
            target
        )

    print("\nInformation Gain:")

    for attribute, gain in gains.items():

        print(
            attribute,
            "=",
            round(gain, 4)
        )

    best = max(
        gains,
        key=gains.get
    )

    print("Best Attribute:", best)

    return best


# --------------------------------------------------
# STEP 5: ID3 Algorithm
# --------------------------------------------------

def id3(data, attributes, target):

    # If all target values are same
    if len(data[target].unique()) == 1:

        return data[target].iloc[0]

    # If no attributes remain
    if len(attributes) == 0:

        return data[target].mode()[0]

    # Find best attribute
    best = best_attribute(
        data,
        attributes,
        target
    )

    tree = {
        best: {}
    }

    # Create branches
    for value in data[best].unique():

        subset = data[
            data[best] == value
        ]

        remaining_attributes = [
            attribute
            for attribute in attributes
            if attribute != best
        ]

        subtree = id3(
            subset,
            remaining_attributes,
            target
        )

        tree[best][value] = subtree

    return tree


# --------------------------------------------------
# STEP 6: Build Decision Tree
# --------------------------------------------------

attributes = [
    'Outlook',
    'Temperature',
    'Humidity',
    'Wind'
]

print("\n\nBUILDING DECISION TREE")

decision_tree = id3(
    df,
    attributes,
    'Play'
)

print("\nFINAL DECISION TREE:")
print(decision_tree)


# --------------------------------------------------
# STEP 7: Prediction Function
# --------------------------------------------------

def predict(tree, sample):

    # If tree has reached a leaf
    if not isinstance(tree, dict):

        return tree

    # Get decision attribute
    attribute = next(iter(tree))

    # Get sample value
    value = sample[attribute]

    # Check whether branch exists
    if value in tree[attribute]:

        return predict(
            tree[attribute][value],
            sample
        )

    else:

        return "Unknown"


# --------------------------------------------------
# STEP 8: Test New Data
# --------------------------------------------------

sample1 = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

sample2 = {
    'Outlook': 'Overcast',
    'Temperature': 'Hot',
    'Humidity': 'High',
    'Wind': 'Strong'
}

sample3 = {
    'Outlook': 'Rain',
    'Temperature': 'Mild',
    'Humidity': 'Normal',
    'Wind': 'Weak'
}


print("\nPREDICTIONS")

print(
    "Sample 1:",
    predict(decision_tree, sample1)
)

print(
    "Sample 2:",
    predict(decision_tree, sample2)
)

print(
    "Sample 3:",
    predict(decision_tree, sample3)
)


DATASET
     Outlook Temperature Humidity    Wind Play
0      Sunny         Hot     High    Weak   No
1      Sunny         Hot     High  Strong   No
2   Overcast         Hot     High    Weak  Yes
3       Rain        Mild     High    Weak  Yes
4       Rain        Cool   Normal    Weak  Yes
5       Rain        Cool   Normal  Strong   No
6   Overcast        Cool   Normal  Strong  Yes
7      Sunny        Mild     High    Weak   No
8      Sunny        Cool   Normal    Weak  Yes
9       Rain        Mild   Normal    Weak  Yes
10     Sunny        Mild   Normal  Strong  Yes
11  Overcast        Mild     High  Strong  Yes
12  Overcast         Hot   Normal    Weak  Yes
13      Rain        Mild     High  Strong   No


BUILDING DECISION TREE

Information Gain:
Outlook = 0.2467
Temperature = 0.0292
Humidity = 0.1518
Wind = 0.0481
Best Attribute: Outlook

Information Gain:
Temperature = 0.571
Humidity = 0.971
Wind = 0.02
Best Attribute: Humidity

Information Gain:
Temperature = 0.02
Humidity = 0.02
Wi